# Depth Completion Prototype — CompletionFormer on nuScenes

Validates the O4 pipeline step: sparse aggregated LiDAR → CompletionFormer → dense metric depth → (H,W,3) pointmap.

**Scope**: no SAM3, no mask filtering. Global pipeline only.

**Steps**:
1. Load nuScenes frame + aggregated pre-filtered LiDAR (existing pipeline)
2. Project to sparse `(H,W)` depth map — min depth per pixel
3. Run CompletionFormer → dense metric depth
4. Back-project → `(H,W,3)` pointmap in PyTorch3D convention
5. Visualise: image | sparse input | completed depth | point cloud overlay

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import cv2
import torch
import matplotlib
import matplotlib.pyplot as plt
from pyquaternion import Quaternion
from nuscenes.nuscenes import NuScenes

print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Config
# ══════════════════════════════════════════════════════════════════════════════
DATASET = 'nuscenes_mini'   # 'nuscenes_mini' or 'ecp'

if DATASET == 'nuscenes_mini':
    DATA_ROOT    = '/media/lleba/ECP_Nuscenes_01/nuScenes_mini'
    NUSCENES_VER = 'v1.0-mini'
    SCENE_IDX    = 3
    FRAME_IDX    = 20    # pick a keyframe for meaningful GT comparison
    CAMERA       = 'CAM_FRONT'
    N_BEFORE, N_AFTER = 3, 3   # sweep aggregation; set 0,0 for single sweep
    LIDAR_LINES  = 32          # nuScenes: 32-beam Velodyne
elif DATASET == 'ecp':
    DATA_ROOT    = '/media/lleba/ECP_Nuscenes_01/output2/ecp2nuscenes'
    NUSCENES_VER = 'v1.0-trainval'
    SCENE_IDX    = 10
    FRAME_IDX    = 560
    CAMERA       = 'CAM_FRONT'
    N_BEFORE, N_AFTER = 2, 2
    LIDAR_LINES  = 64          # ECP: 64-beam LiDAR
else:
    raise ValueError(f'Unknown DATASET: {DATASET!r}')

DEV_ROOT    = Path('/home/lleba/Thesis/Development')
CFORMER_ROOT = DEV_ROOT / 'Models' / 'CompletionFormer'
CFORMER_CKPT = CFORMER_ROOT / 'checkpoints' / 'KITTIDC' / 'KITTIDC_L1L2.pt'

# LiDAR pre-filters (same values as full pipeline)
USE_EGO_BODY_FILTER = True
EGO_BOX_HALF_X = 4.0
EGO_BOX_HALF_Y = 1.5
EGO_BOX_Z_MIN  = 0.5
EGO_BOX_Z_MAX  = 2.5
MAX_LIDAR_RANGE_M = 52.0

print(f'Dataset : {DATASET}  scene={SCENE_IDX}  frame={FRAME_IDX}  camera={CAMERA}')
print(f'LiDAR   : {LIDAR_LINES}-beam')
print(f'Aggregation: N_BEFORE={N_BEFORE}  N_AFTER={N_AFTER}  (set both to 0 for single sweep)')
print(f'Checkpoint : {CFORMER_CKPT}')
print(f'Exists     : {CFORMER_CKPT.exists()}')


In [ ]:
# Add autolabeling src so we can reuse existing LiDAR loaders and filters
AL_SRC = str(DEV_ROOT / 'AutoLabeling' / 'src')
if AL_SRC not in sys.path:
    sys.path.insert(0, AL_SRC)

from autolabeling.data.nuscenes_loader import FrameRecord
from autolabeling.utils.lidar import load_lidar_pts, load_lidar_pts_aggregated, filter_lidar_pts
print('Autolabeling utils loaded.')


In [ ]:
nusc = NuScenes(version=NUSCENES_VER, dataroot=DATA_ROOT, verbose=False)

scene   = nusc.scene[SCENE_IDX]
token   = nusc.get('sample', scene['first_sample_token'])['data'][CAMERA]
for _ in range(FRAME_IDX):
    rec   = nusc.get('sample_data', token)
    token = rec['next']
    assert token, f'FRAME_IDX {FRAME_IDX} exceeds scene length'

rec      = nusc.get('sample_data', token)
img_path = nusc.get_sample_data_path(token)
cal      = nusc.get('calibrated_sensor', rec['calibrated_sensor_token'])

K     = np.array(cal['camera_intrinsic'], dtype=np.float64)
R_c2e = Quaternion(cal['rotation']).rotation_matrix.astype(np.float64)
t_c2e = np.array(cal['translation'], dtype=np.float64)

img_bgr = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
H, W    = img_rgb.shape[:2]
cam_sd_token = token

ep      = nusc.get('ego_pose', rec['ego_pose_token'])
R_e2g   = Quaternion(ep['rotation']).rotation_matrix.astype(np.float64)
t_e2g   = np.array(ep['translation'], dtype=np.float64)

print(f'Image: {W}x{H}  path: {img_path}')
print(f'K[0,0]={K[0,0]:.1f}  K[1,1]={K[1,1]:.1f}  cx={K[0,2]:.1f}  cy={K[1,2]:.1f}')
plt.figure(figsize=(14,4)); plt.imshow(img_rgb); plt.axis('off')
plt.title(f'Scene {SCENE_IDX} ({scene["name"]}), frame {FRAME_IDX} — {CAMERA}'); plt.show()


In [ ]:
# ── Build FrameRecord (lidar_sd_token needed for aggregation) ────────────────
_sd_rec = nusc.get('sample_data', cam_sd_token)
_sample = nusc.get('sample', _sd_rec['sample_token'])
_lidar_sd_token = _sample['data']['LIDAR_TOP']
_lidar_sd  = nusc.get('sample_data', _lidar_sd_token)
_lidar_cs  = nusc.get('calibrated_sensor', _lidar_sd['calibrated_sensor_token'])
_R_l2e = Quaternion(_lidar_cs['rotation']).rotation_matrix.astype(np.float64)
_t_l2e = np.array(_lidar_cs['translation'], dtype=np.float64)
_lidar_path = str(Path(DATA_ROOT) / _lidar_sd['filename'])

frame = FrameRecord(
    sample_token   = _sample['token'],
    img_path       = img_path,
    K              = K,
    R_c2e          = R_c2e,
    t_c2e          = t_c2e,
    R_e2g          = R_e2g,
    t_e2g          = t_e2g,
    scene_name     = scene['name'],
    frame_idx      = FRAME_IDX,
    lidar_path     = _lidar_path,
    R_l2e          = _R_l2e,
    t_l2e          = _t_l2e,
    lidar_sd_token = _lidar_sd_token,
)

# ── Load aggregated LiDAR ─────────────────────────────────────────────────────
if N_BEFORE > 0 or N_AFTER > 0:
    pts_ego = load_lidar_pts_aggregated(nusc, frame, N_BEFORE, N_AFTER)
    print(f'Aggregated ({N_BEFORE}+1+{N_AFTER} sweeps): {len(pts_ego):,} points')
else:
    pts_ego = load_lidar_pts(frame)
    print(f'Single sweep: {len(pts_ego):,} points')

# ── Global pre-filters ────────────────────────────────────────────────────────
pts_ego = filter_lidar_pts(
    pts_ego,
    use_ego_body_filter=USE_EGO_BODY_FILTER,
    ego_box_half_x=EGO_BOX_HALF_X,
    ego_box_half_y=EGO_BOX_HALF_Y,
    ego_box_z_min=EGO_BOX_Z_MIN,
    ego_box_z_max=EGO_BOX_Z_MAX,
    max_range_m=MAX_LIDAR_RANGE_M,
)
print(f'After pre-filters: {len(pts_ego):,} points')


In [ ]:
# ── Project ego-frame LiDAR into camera ──────────────────────────────────────
# ego → camera:  R_c2e^T @ (p - t_c2e)
pts_cam = (R_c2e.T @ (pts_ego[:, :3] - t_c2e).T).T  # (N, 3) in camera frame

# Keep only points in front of camera
valid   = pts_cam[:, 2] > 0.1
pts_cam = pts_cam[valid]

# Project to pixel coordinates
fx, fy, cx, cy = K[0,0], K[1,1], K[0,2], K[1,2]
u = pts_cam[:, 0] / pts_cam[:, 2] * fx + cx
v = pts_cam[:, 1] / pts_cam[:, 2] * fy + cy
z = pts_cam[:, 2]

# Keep points inside image bounds
in_img = (u >= 0) & (u < W) & (v >= 0) & (v < H)
u_int = np.clip(np.round(u[in_img]).astype(int), 0, W - 1)
v_int = np.clip(np.round(v[in_img]).astype(int), 0, H - 1)
z_vis = z[in_img]

# Build sparse depth map — min depth per pixel (foreground wins)
sparse_depth = np.zeros((H, W), dtype=np.float32)
# Use large init so min works correctly
_tmp = np.full((H, W), np.inf, dtype=np.float32)
np.minimum.at(_tmp, (v_int, u_int), z_vis.astype(np.float32))
sparse_depth[_tmp < np.inf] = _tmp[_tmp < np.inf]

n_pixels = (sparse_depth > 0).sum()
coverage = 100 * n_pixels / (H * W)
print(f'Sparse depth: {n_pixels:,} valid pixels / {H*W:,} total = {coverage:.2f}% coverage')
print(f'Depth range: {z_vis.min():.1f} – {z_vis.max():.1f} m')

# Visualise sparse depth
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
axes[0].imshow(img_rgb); axes[0].axis('off'); axes[0].set_title('RGB')
d_vis = np.where(sparse_depth > 0, sparse_depth, np.nan)
im = axes[1].imshow(d_vis, cmap='turbo', vmin=0, vmax=60)
plt.colorbar(im, ax=axes[1], label='depth (m)')
axes[1].axis('off'); axes[1].set_title(f'Sparse LiDAR depth ({coverage:.1f}% coverage)')
plt.tight_layout(); plt.show()


In [ ]:
# ── Load CompletionFormer ─────────────────────────────────────────────────────
CFORMER_SRC = str(CFORMER_ROOT / 'src')
CFORMER_DCN = str(CFORMER_ROOT / 'src' / 'model' / 'deformconv')
for _p in [CFORMER_SRC, CFORMER_DCN]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# Build args namespace directly — avoids importing config.py which calls
# parse_args() at module level and picks up Jupyter kernel sys.argv
import argparse
args = argparse.Namespace(
    data_name       = 'KITTIDC',
    prop_time       = 6,
    prop_kernel     = 3,
    preserve_input  = False,   # set True to hard-anchor LiDAR values at anchor pixels
    affinity        = 'TGASS',
    affinity_gamma  = 0.5,
    conf_prop       = True,
    legacy          = False,
    from_scratch    = False,
    max_depth       = 90.0,
    num_sample      = 500,
    lidar_lines     = LIDAR_LINES,
)

from model.completionformer import CompletionFormer

cformer = CompletionFormer(args).cuda()

ckpt = torch.load(str(CFORMER_CKPT), map_location='cuda', weights_only=False)
state = ckpt.get('net', ckpt)  # checkpoint may wrap weights under 'net' key
cformer.load_state_dict(state, strict=True)
cformer.eval()
print(f'CompletionFormer loaded from {CFORMER_CKPT.name}')
print(f'Parameters: {sum(p.numel() for p in cformer.parameters())/1e6:.1f} M')
print(f'preserve_input={args.preserve_input}  (set True to hard-anchor LiDAR values)')


In [ ]:
# ── Prepare inputs ────────────────────────────────────────────────────────────
# RGB: (1, 3, H, W)  float32  0-1
rgb_t = torch.from_numpy(img_rgb.astype(np.float32) / 255.0)
rgb_t = rgb_t.permute(2, 0, 1).unsqueeze(0).cuda()  # (1,3,H,W)

# Sparse depth: (1, 1, H, W)  float32  metres
dep_t = torch.from_numpy(sparse_depth).unsqueeze(0).unsqueeze(0).cuda()  # (1,1,H,W)

# ── Run CompletionFormer ──────────────────────────────────────────────────────
with torch.no_grad():
    output = cformer({'rgb': rgb_t, 'dep': dep_t})

dense_depth = output['pred'].squeeze().cpu().numpy()  # (H, W)  metres
print(f'Dense depth: shape={dense_depth.shape}  range={dense_depth.min():.2f}–{dense_depth.max():.2f} m')
print(f'Mean depth: {dense_depth[dense_depth>0].mean():.2f} m')


In [ ]:
# ── Visualise: RGB | sparse | completed | error (at LiDAR pixels) ────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 9))

axes[0,0].imshow(img_rgb); axes[0,0].axis('off'); axes[0,0].set_title('RGB')

d_sparse_vis = np.where(sparse_depth > 0, sparse_depth, np.nan)
im1 = axes[0,1].imshow(d_sparse_vis, cmap='turbo', vmin=0, vmax=60)
plt.colorbar(im1, ax=axes[0,1], label='m')
axes[0,1].axis('off'); axes[0,1].set_title(f'Sparse LiDAR input ({coverage:.1f}% coverage)')

im2 = axes[1,0].imshow(dense_depth, cmap='turbo', vmin=0, vmax=60)
plt.colorbar(im2, ax=axes[1,0], label='m')
axes[1,0].axis('off'); axes[1,0].set_title('CompletionFormer output (dense metric depth)')

# Error at LiDAR anchor pixels
mask  = sparse_depth > 0
error = np.abs(dense_depth - sparse_depth)
err_vis = np.where(mask, error, np.nan)
im3 = axes[1,1].imshow(err_vis, cmap='hot', vmin=0, vmax=1.0)
plt.colorbar(im3, ax=axes[1,1], label='|pred - lidar| (m)')
axes[1,1].axis('off')
mae = float(error[mask].mean())
axes[1,1].set_title(f'Abs error at LiDAR pixels (MAE={mae:.3f} m)')

plt.suptitle(f'Scene {SCENE_IDX}, frame {FRAME_IDX} — {CAMERA}  |  N_BEFORE={N_BEFORE} N_AFTER={N_AFTER}', y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# ── Back-project dense depth → (H,W,3) pointmap in PyTorch3D convention ──────
# PyTorch3D: (-X_cam, -Y_cam, Z_cam)
v_grid, u_grid = np.mgrid[0:H, 0:W]
Z = dense_depth.astype(np.float64)
X =  (u_grid - cx) / fx * Z
Y =  (v_grid - cy) / fy * Z

# PyTorch3D convention: negate X and Y
ptmap = np.stack([-X, -Y, Z], axis=-1).astype(np.float32)  # (H, W, 3)
print(f'Pointmap shape: {ptmap.shape}')
print(f'X range: {(-X).min():.2f}–{(-X).max():.2f}')
print(f'Y range: {(-Y).min():.2f}–{(-Y).max():.2f}')
print(f'Z range: {Z.min():.2f}–{Z.max():.2f}')
print('Pointmap ready for SAM3D Objects inference.')


In [ ]:
# ── 3D point cloud: completed depth reprojected to ego frame vs raw LiDAR ────
import plotly.graph_objects as go

Z_f  = dense_depth.astype(np.float32).ravel()
valid_px = Z_f > 0.1
u_all = (u_grid.ravel()[valid_px] - cx) / fx
v_all = (v_grid.ravel()[valid_px] - cy) / fy
z_all = Z_f[valid_px]
pts_cam_dense = np.stack([u_all * z_all, v_all * z_all, z_all], axis=1)

# Camera → ego
pts_ego_dense = (R_c2e @ pts_cam_dense.T).T + t_c2e  # (N,3) ego

# Subsample for plotting
rng   = np.random.default_rng(0)
idx_d = rng.choice(len(pts_ego_dense), min(40_000, len(pts_ego_dense)), replace=False)
idx_l = rng.choice(len(pts_ego),       min(15_000, len(pts_ego)),       replace=False)

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=pts_ego_dense[idx_d, 0], y=pts_ego_dense[idx_d, 1], z=pts_ego_dense[idx_d, 2],
    mode='markers',
    marker=dict(size=1.5, color=pts_ego_dense[idx_d, 2], colorscale='turbo',
                cmin=-2, cmax=3, colorbar=dict(title='Z (m)'), showscale=True),
    name='completed depth',
    hovertemplate='X=%{x:.1f} Y=%{y:.1f} Z=%{z:.2f}<extra>completed</extra>',
))
fig.add_trace(go.Scatter3d(
    x=pts_ego[idx_l, 0], y=pts_ego[idx_l, 1], z=pts_ego[idx_l, 2],
    mode='markers',
    marker=dict(size=2, color='white', opacity=0.6),
    name='LiDAR',
    hovertemplate='X=%{x:.1f} Y=%{y:.1f} Z=%{z:.2f}<extra>LiDAR</extra>',
))
fig.update_layout(
    title=f'3D point cloud — completed depth vs LiDAR  |  scene {SCENE_IDX} frame {FRAME_IDX}',
    scene=dict(
        xaxis=dict(title='X ego (forward, m)', range=[-5, 60]),
        yaxis=dict(title='Y ego (left, m)',    range=[-30, 30]),
        zaxis=dict(title='Z ego (up, m)',      range=[-3, 5]),
        bgcolor='#0f0f19',
        aspectmode='manual',
        aspectratio=dict(x=3, y=2, z=0.3),
    ),
    paper_bgcolor='#0f0f19',
    font=dict(color='white'),
    legend=dict(bgcolor='#222'),
    height=600,
    margin=dict(l=0,r=0,t=40,b=0),
)
fig.show()
